In [1]:
from radpy.sedfit import *
from radpy.stellar import *
from radpy.sed_batch_mode import *
#import ast
#from radpy.batchmode import extract_id, find_files_for_star, convert_names_to_latex, format_catalog_name, save_plot

/home/oxfor/miniforge/envs/test_env/lib/python3.12/site-packages/radpy/sedfit.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Holoviews not imported. Some visualizations will not be available.
PyMultiNest not imported.  MultiNest fits will not work.
/home/oxfor/miniforge/envs/test_env/lib/python3.12/site-packages/astroARIADNE/fitter.py:43: UserWarning: (py)MultiNest installation (or libmultinest.dylib) not detected.
  warnings.warn(


/home/oxfor/miniforge/envs/test_env/lib/python3.12/site-packages/radpy/sedfit.py:819: SyntaxWarning: invalid escape sequence '\s'
  phot = pd.read_csv(filename, sep='\s+', skiprows=1, header=None)


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
data_dir = "/mnt/c/Users/oxfor/Research/rsadpy/tests/tests/test_data/photometry"
param_file = '/mnt/c/Users/oxfor/Research/rsadpy/tests/test_data/photometry/allphot_test.csv'
out_dir = "/mnt/c/Users/oxfor/Research/rsadpy/tests/tests/test_data/photometry"
res_out = "SEDresults.txt"
diam_out = "stardatafordiams.txt"
unit = 'AA'
set_axis = None
image_ext = '.jpg'
uselatex = False
logplot = True
fbol_lam = True
own_photometry = True
sed_batchmode(param_file, data_dir, out_dir, res_out, diam_out, unit, set_axis, image_ext, uselatex,
              logplot, fbol_lam, own_photometry, verbose = True)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Disable GPU
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  #suppress everything but error messages
import tensorflow as tf

# Print devices to check if GPU is being used
print("GPU devices available: ", tf.config.list_physical_devices("GPU"))

In [ ]:
def setaxislabels(exp, unit, logplot = False, fbol_lam = False):
    ##########################################################
    # Function: setaxislabels                                #
    # Inputs:                                                #
    #    unit: string of unit wanted                         #
    #    fbol_lam: flag for fbol_lam                         #
    # Outputs:                                               #
    #    xlab: x axis label                                  #
    #    ylab: y axis label                                  #
    # How it works:                                          #
    #    1. Based on unit chosen, sets x label dependent on  #
    #       unit                                             #
    #    2. Based on unit chosen and if the fbol_lam flag    #
    #       has been set, sets y label                       #
    #    3. Returns x axis and y axis labels                 #
    ##########################################################
    if logplot:
        if fbol_lam:
            ylab = r'$\rm \lambda F_{\lambda}~[\frac{erg}{cm^{2}~s}$]'
        else:
            if unit == 'AA':
                ylab = r'$\rm F_{\lambda}~[\frac{erg}{cm^{2}~s~\AA}$]'
            elif unit == 'micron':
                ylab = r'$\rm F_{\lambda}~[\frac{erg}{cm^{2}~s~\mu m}$]'
        if unit == 'AA':
            xlab = r'$\rm Wavelength~[\AA]$'
        elif unit == 'micron':
            xlab = r'$\rm Wavelength~[\mu m]$'
        return xlab, ylab
    else:
        if fbol_lam:
            #print('Fbol lam')
            #print('Exponent:', exp)
            if exp < 0:
                #print('Exp < 0')
                ylab = rf'$\rm \lambda F_{{\lambda}}~[\times 10^{{{exp}}}~\frac{{\rm erg}}{{\rm cm^2~s}}]$'
            if unit == 'AA':
                #print('angstroms')
                xlab = r'$\rm Wavelength~[\AA]$'
            elif unit == 'micron':
                #print('Microns')
                xlab = r'$\rm Wavelength~[\mu m]$'
            return xlab, ylab
        else:
            #print('No fbol lam')
            #print('Exoponent:', exp)
            if unit == 'AA':
                #print('Angstroms')
                xlab = r'$\rm Wavelength~[\AA]$'
                if exp < 0:
                    #print('exp < 0')
                    ylab = rf'$\rm F_{{\lambda}}~[\times 10^{{{exp}}}~\frac{{\rm erg}}{{\rm cm^2~s~\AA}}]$'
            elif unit == 'micron':
                #print('Microns')
                xlab = r'$\rm Wavelength~[\mu m]$'
                if exp < 0:
                    #print('Exp < 0')
                    ylab = rf'$\rm F_{{\lambda}}~[\times 10^{{{exp}}}~\frac{{\rm erg}}{{\rm cm^2~s~\mu m}}]$'
            return xlab, ylab

In [ ]:
def set_values(x, unit, logplot=False, fbol_lam=False, verbose=False):
    ##########################################################
    # Function: set_values                                   #
    # Inputs:                                                #
    #    x: sed object                                       #
    #    unit: unit string                                   #
    #    logPlot: default is False, if True, indicates the   #
    #             log flag                                   #
    #    fbol_lam: default is False, if True, indicates the  #
    #             fbol_lam is flag                           #
    # Outputs:                                               #
    #    litp_xvals: input wavelength                        #
    #    litp_yvals: input flux                              #
    #    litp_dxvals: input wavelength error                 #
    #    litp_dyvals: input flux error                       #
    #    model_xvals: model wavelength                       #
    #    model_yvals: model flux                             #
    #    synth_yvals: model fluxes in the wavelength         #
    #                 bandpasses                             #
    #   residuals: lit flux values minus synth values        #
    # How it works:                                          #
    #    1. Calls convert to generate the model values       #
    #    2. Based on the flags set, converts the values to   #
    #       match what the flags say                         #
    #       logplot: convert everything back into log10      #
    #       fbol_lam: multiply the flux by wavelength        #
    #    3. Returns values                                   #
    ##########################################################
    iwave, iflux, idwave, idflux, mw, mf, msf = convert(x, unit=unit)

    if logplot and fbol_lam:
        # print('set values: Log and lambda')
        model_xvals = np.log10(mw)
        model_yvals = np.log10(mf * mw)
        litp_xvals = np.log10(iwave)
        litp_yvals = np.log10(iwave * iflux)
        litp_dyvals = 0.434 * (idflux / iflux)
        litp_dxvals = 0.434 * (idwave / iwave)
        synth_yvals = np.log10(msf * iwave)
        res = litp_yvals - synth_yvals
        exp = 0
        
        return model_xvals, model_yvals, litp_xvals, litp_yvals, litp_dxvals, litp_dyvals, synth_yvals, res, exp

    if logplot and not fbol_lam:
        # print('set values Log and no lambda')
        model_xvals = np.log10(mw)
        model_yvals = np.log10(mf)
        litp_xvals = np.log10(iwave)
        litp_yvals = np.log10(iflux)
        litp_dyvals = 0.434 * (idflux / iflux)
        litp_dxvals = 0.434 * (idwave / iwave)
        synth_yvals = np.log10(msf)
        res = litp_yvals - synth_yvals
        exp = 0
        return model_xvals, model_yvals, litp_xvals, litp_yvals, litp_dxvals, litp_dyvals, synth_yvals, res, exp

    if not logplot and not fbol_lam:
        # print('set values No log and no lambda')
        number = iflux[0]
        _, exp = normalize_number(number)
        # print('Dividing by:', exp)
        model_xvals = mw
        model_yvals = mf / (10 ** exp)
        litp_xvals = iwave
        litp_yvals = iflux / (10 ** exp)
        litp_dyvals = idflux / (10 ** exp)
        litp_dxvals = idwave
        synth_yvals = msf / (10 ** exp)
        res = litp_yvals - synth_yvals

        return model_xvals, model_yvals, litp_xvals, litp_yvals, litp_dxvals, litp_dyvals, synth_yvals, res, exp

    if not logplot and fbol_lam:
        number = (iflux[0] * iwave[0])
        _, exp = normalize_number(number)
        # print('Dividing by:', exp)
        model_xvals = mw
        model_yvals = (mf * mw) / (10 ** exp)
        litp_xvals = iwave
        litp_yvals = (iflux * iwave) / (10 ** exp)
        litp_dyvals = (idflux) / (10 ** exp)
        litp_dxvals = idwave
        synth_yvals = (msf * iwave) / (10 ** exp)
        res = litp_yvals - synth_yvals

        return model_xvals, model_yvals, litp_xvals, litp_yvals, litp_dxvals, litp_dyvals, synth_yvals, res, exp

In [ ]:
def set_res_axis(res, logplot = False):
    min_res = np.min(res)
    max_res = np.max(res)

    minres = round(min_res, 1)
    maxres = round(max_res, 1)

    if abs(minres) >= abs(maxres):
        if logplot:
            res_axis_min = (abs(minres)+0.05)*-1
            res_axis_max = abs(minres)+0.05
            res_loc = [abs(minres)*-1, 0, abs(minres)]
            res_labels = [rf'$\rm {val}$' for val in res_loc]
        else:
            res_axis_min = (abs(minres)+0.5)*-1
            res_axis_max = abs(minres)+0.5
            res_loc = [round(abs(minres)*-1), 0, round(abs(minres))]
            res_labels = [rf'$\rm {val}$' for val in res_loc]
    elif abs(minres) <= abs(maxres):
        if logplot:
            res_axis_min = (abs(maxres)+0.05)*-1
            res_axis_max = abs(maxres)+0.05
            res_loc = [abs(maxres)*-1, 0, abs(maxres)]
            res_labels = [rf'$\rm {val}$' for val in res_loc]
        else:
            res_axis_min = (abs(maxres)+0.5)*-1
            res_axis_max = abs(maxres)+0.5
            res_loc = [round(abs(maxres)*-1), 0, round(abs(maxres))]
            res_labels = [rf'$\rm {val}$' for val in res_loc]

    return res_axis_min, res_axis_max, res_loc, res_labels

In [ ]:
def setaxisticklabels(iwave, iflux, exp, unit, set_axis, logplot=False, fbol_lam=False, verbose=False):
    if set_axis:
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]
    elif set_axis is None:
        set_axis = set_radpy_axis_limits(iwave, iflux, exp, unit, logplot = logplot)
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]
        
    if logplot:
        xmin = round((xmin), 1)
        xmax = round((xmax), 1)
        ymin = round(ymin)
        ymax = round(ymax)
        # y ticks
        y_loc = [ymin, ymax]
        y_labels = [rf'$10^{{{y_loc[0]}}}$', rf'$10^{{{y_loc[1]}}}$']
            
        # x ticks
        xaxis = np.linspace(xmin, xmax, 5)
        #xaxis is in np.log10 space
        x_loc = []
        xl = []
        for i in range(len(xaxis)):
            #converting out of log10 space
            xloc = (10 ** (xaxis[i]))
            if unit == 'AA':
                #xlabel is out of log10 space
                xl.append(int((round(xloc, -3))))
                #xlocation is in log10 space
                x_loc.append(np.log10((round(xloc, -3))))
            if unit == 'micron':
                if xloc < 0.8:
                    #xlabel is out of log10 space
                    xl.append(round(xloc, 1))
                    #xlocation is in log10 space
                    x_loc.append(np.log10(round(xloc, 1)))
                else:
                    #xlabel is out of log10 space
                    xl.append(round(xloc))
                    #xlocation is in log10 space
                    x_loc.append(np.log10(round(xloc)))

        x_labels = [rf'$\rm {(val)}$' for val in xl]
        return x_loc, x_labels, y_loc, y_labels
    else:
        #y axis limits are in 10^exp space
        if fbol_lam:
            # y ticks
            #taking the yvals out of 10^exp space
            yloc = np.linspace(ymin / (10 ** exp), ymax / (10 ** exp), 4)
            y_loc = [round(val) for val in yloc]
            y_labels = [rf'$\rm {round(val)}$' for val in y_loc]

            # x ticks
            xaxis = np.linspace(xmin, xmax, 5)
            x_loc = []
            xl = []
            for i in range(len(xaxis)):
                xloc = xaxis[i]
                if unit == 'AA':
                    xl.append(int((round(xloc, -3))))
                    x_loc.append((round(xloc, -3)))
                if unit == 'micron':
                    if xloc < 1 and xloc > 0:
                        xl.append(round(xloc,2))
                        x_loc.append(round(xloc,2))
                    elif xloc == 0:
                        xl.append(round(xloc))
                        x_loc.append(round(xloc))
                    else:
                        if xloc%1 < 0.5:
                            xl.append(round(np.floor(xloc)))
                            x_loc.append(round(np.floor(xloc)))
                        elif xloc%1 > 0.5:
                            xl.append(round((xloc)))
                            x_loc.append(round((xloc)))
                            
            x_labels = [rf'${(val)}$' for val in xl]
            return x_loc, x_labels, y_loc, y_labels
        else:
            # y ticks
            yloc = np.linspace(1, ymax / (10 ** exp), 4)
            y_loc = [round(val) for val in yloc]
            y_labels = [rf'$ \rm {round(val)}$' for val in y_loc]
            # x ticks
            xaxis = np.linspace(xmin, xmax, 5)
            x_loc = []
            xl = []
            for i in range(len(xaxis)):
                xloc = xaxis[i]
                if unit == 'AA':
                    xl.append(int((round(xloc, -3))))
                    x_loc.append((round(xloc, -3)))
                if unit == 'micron':
                    if xloc < 1 and xloc > 0:
                        xl.append(round(xloc,2))
                        x_loc.append(round(xloc,2))
                    elif xloc == 0:
                        xl.append(round(xloc))
                        x_loc.append(round(xloc))
                    else:
                        if xloc%1 < 0.5:
                            xl.append(round(np.floor(xloc)))
                            x_loc.append(round(np.floor(xloc)))
                        elif xloc%1 > 0.5:
                            xl.append(round((xloc)))
                            x_loc.append(round((xloc)))
                            
            x_labels = [rf'${(val)}$' for val in xl]
            return x_loc, x_labels, y_loc, y_labels

In [ ]:
def set_radpy_axis_limits(w, f, exp, unit, logplot):
    ##########################################################
    # Function: set_axis_labels                              #
    # Inputs:                                                #
    #    mw: model wavelength array                          #
    #    mf: model flux array                                #
    #    unit: string of unit wanted                         #
    #    fbol_lam: flag for fbol_lam                         #
    # Outputs:                                               #
    #    set_axis: the axis limits in the format of          #
    #              [xmin, xmax, ymin, ymax]                  #
    # How it works:                                          #
    #    1. Based on unit chosen and fbol_lam flag setting,  #
    #       sets the axis limits based on the minimum of the #
    #       arrays  and maxs of the arrays                   #
    #    2. Returns the sxis limits                          #
    ##########################################################
    xmin = min(w)
    xmax = max(w)
    ymin = min(f)
    ymax = max(f)

    
    if unit == 'AA':
        if logplot:
            Xmin = xmin-(xmin*0.05)
            Xmax = xmax+(xmax*0.05)
            Ymin = ymin+(ymin*0.05)
            Ymax = ymax-(ymax*0.05)

            new_axis = [round(Xmin, 1),round(Xmax, 1), round(Ymin)*(10**(exp)), round(Ymax)*(10**(exp))]

            return new_axis
        else:
            Xmin = xmin-(xmin*0.1)
            Xmax = xmax+(xmax*0.1)
            Ymin = ymin+(ymin*0.1)
            Ymax = ymax+(ymax*0.2)

            new_axis = [round(Xmin, -2), round(Xmax,-3), round(Ymin, 1)*(10**(exp)), round(Ymax)*(10**(exp))]

            return new_axis
    if unit == 'micron':
        if logplot:
            Xmin = xmin-(xmin*0.05)
            Xmax = xmax+(xmax*0.05)
            Ymin = ymin+(ymin*0.01)
            Ymax = ymax-(ymax*0.01)
            
            new_axis = [round(Xmin, 2),round(Xmax, 2), round(Ymin)*(10**(exp)), round(Ymax)*(10**(exp))]

            return new_axis
        else:
            Xmin = xmin-(xmin*0.05)
            Xmax = xmax+(xmax*0.05)
            Ymin = ymin-(ymin*0.05)
            Ymax = ymax+(ymax*0.1)
            
            new_axis = [round(Xmin, 2), round(Xmax,2), round(Ymin, 1)*(10**(exp)), round(Ymax,1)*(10**(exp))]

            return new_axis

In [ ]:
def setaxislimits(iwave, iflux, exp, unit, set_axis, logplot=False, fbol_lam=False):
    if set_axis:
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]
    else:
        set_axis = set_radpy_axis_limits(iwave, iflux, exp, unit, logplot = logplot)
        xmin = set_axis[0]
        xmax = set_axis[1]
        ymin = set_axis[2]
        ymax = set_axis[3]
        
    if logplot:
        return xmin - 0.1, xmax + 0.1, ymin - 0.5, ymax + 0.1
    else:
        if unit =='AA':
            return xmin - 500, xmax + 500, (ymin / (10 ** (exp))) - 0.25, (ymax / (10 ** (exp))) + 0.25
        if unit == 'micron':
            return xmin - 0.1, xmax + 0.1, (ymin / (10 ** (exp))) - 0.25, (ymax / (10 ** (exp))) + 0.25


In [ ]:
def plot_sed(x, unit, logplot=True, fbol_lam=True, set_axis=None, title=None, savefig=None, uselatex = False, show=True, verbose=False):
    ##########################################################
    # Function: plot_sed                                     #
    # Inputs:                                                #
    #    x: sed object                                       #
    #    unit: unit chosen                                   #
    #    logplot: log flag                                   #
    #             if True, sets plot in log space            #
    #    fbol_lam: fbol_lam flag                             #
    #             if True, multiplies the flux by wavelength #
    #    set_axis: allows user to set their axis limits      #
    #             if None, will set based on the model vals  #
    #    title: allows user to set the plot title            #
    #    savefig: allows user to save fig                    #
    #            give a filename                             #
    #    show: shows Figure                                  #
    # Outputs:                                               #
    #    displays the plot                                   #
    # How it works:                                          #
    #    1. Determines the axis limits based on user input   #
    #       and flags set                                    #
    #    2. Calls set_values to generate the data to be      #
    #       plotted                                          #
    #    3. Plots everything                                 #
    ##########################################################

    #iwave, iflux, idwave, idflux, mw, mf, msf = convert(x, unit)

    plt.rcParams.update({'font.size': 15})
    plt.rcParams['xtick.direction'] = 'in'
    plt.rcParams['ytick.direction'] = 'in'
    plt.rcParams['text.usetex'] = uselatex

    f, axes = plt.subplots(2, 1, gridspec_kw={'height_ratios': [10, 3]}, sharex=True)

    model_xvals, model_yvals, litp_xvals, litp_yvals, litp_dxvals, litp_dyvals, synth_yvals, res, exp = set_values(x, unit,
                                                                                                                   logplot=logplot,
                                                                                                                   fbol_lam=fbol_lam,
                                                                                                                   verbose=verbose)
    xmin, xmax, ymin, ymax = setaxislimits(litp_xvals, litp_yvals, exp, unit, set_axis, logplot=logplot, fbol_lam=fbol_lam)
    xloc, xlabels, yloc, ylabels = setaxisticklabels(litp_xvals, litp_yvals, exp, unit, set_axis, logplot=logplot, fbol_lam=fbol_lam)

    axes[0].set_ylim(ymin, ymax)
    axes[1].set_xlim(xmin, xmax)
    axes[0].set_yticks(yloc)
    axes[0].set_yticklabels(ylabels)
    axes[1].set_xticks(xloc)
    axes[1].set_xticklabels(xlabels)

    axes[0].plot(model_xvals, model_yvals, 'g', linewidth=1, label=r'$\rm Model~Spectrum$')
    axes[0].plot(litp_xvals, litp_yvals, 'b.', markersize=10, markerfacecolor='none', label=r'$\rm Photometry$')
    axes[0].errorbar(litp_xvals, litp_yvals, xerr=litp_dxvals, yerr=litp_dyvals, fmt='.', markerfacecolor='none',
                     color='blue', capsize=3)
    axes[0].plot(litp_xvals, synth_yvals, 'r.', markersize=10, label=r'$\rm Synthetic~Photometry $')

    axes[0].legend(prop={'size': 10}, loc='best')
    axes[0].tick_params(axis='x', labelbottom=False)

    axes[1].plot(litp_xvals, res, 'k.')
    axes[1].errorbar(litp_xvals, res, yerr=litp_dyvals, fmt='.', color='black')
    axes[1].axhline(y=0)
    ramin, ramax, rloc, rlabel = set_res_axis(res, logplot = logplot)
    axes[1].set_ylim(ramin, ramax)
    axes[1].set_yticks(rloc)
    axes[1].set_yticklabels(rlabel)
    if logplot:
        if unit == 'micron':
            axes[1].set_ylabel(r'$\rm Residuals$', labelpad=5)
        if unit == 'AA':
            axes[1].set_ylabel(r'$\rm Residuals$', labelpad=5)
    else:
        if unit == 'micron':
            axes[1].set_ylabel(r'$\rm Residuals$', labelpad=0)
        if unit == 'AA':
            axes[1].set_ylabel(r'$\rm Residuals$', labelpad=0)

    xlab, ylab = setaxislabels(exp, unit, logplot=logplot, fbol_lam=fbol_lam)
    axes[1].set_xlabel(xlab)
    axes[0].set_ylabel(ylab)
    plt.subplots_adjust(wspace=0, hspace=0)
    axes[0].xaxis.set_minor_locator(AutoMinorLocator())
    axes[0].yaxis.set_minor_locator(AutoMinorLocator())
    axes[1].xaxis.set_minor_locator(AutoMinorLocator())
    axes[1].yaxis.set_minor_locator(AutoMinorLocator())

    if title:
        axes[0].set_title(title)
    if savefig:
        f.savefig(savefig, bbox_inches='tight')
    if show:
        plt.show()

    return f, axes

In [ ]:
def get_stellar_params(file_path):
    ######################################################
    # Function: get_stellar_params                       #
    # Inputs: file_path -> path to file                  #
    # Outputs: star_names -> names of the stars          #
    #          star_params -> dictionary of stellar      #
    #                         params                     #
    # What it does:                                      #
    #     1. reads in the file                           #
    #     2. extracts the star names                     #
    #     3. For each star name, extracts the stellar    #
    #        parameters and adds it to the dictionary    #
    #     4. Returns the star names and the dictionary   #
    ######################################################
    df = pd.read_csv(file_path)
    star_names = df['Star'].tolist()
    params_dict = {}
    fitting_dict = {}
    for _, row in df.iterrows():
        params_dict[row['Star']] = {
            'logg': row['logg'],
            'logg_err': row['dlogg'],
            'feh': row['feh'],
            'feh_err': row['dfeh'],
        }
    for _, row in df.iterrows():
        fitting_dict[row['Star']] = {
            'fitTeff': row['fitteff'], 
            'Trange': row['fitteff_range'],
            'fitLogg': row['fitlogg'],
            'Loggrange':row['fitlogg_range'],
            'fitFeh': row['fitfeh'],
            'Fehrange': row['fitfeh_range'],
            'fitAv': row['fitav'],
            'Avrange': row['fitav_range'],
            'model': row['model'],
        }
    return star_names, params_dict, fitting_dict

In [ ]:
def convert_fit_params(star_name, star, fit_params_dict, verbose = False):
    fits = fit_params_dict.get(star_name, {})
    
    fitT = fits['fitTeff']
    fitLG = fits['fitLogg']
    fitFEH = fits['fitFeh']
    fitAV = fits['fitAv']

    model = fits['model']
    
    init_logg = star.logg
    init_feh = star.feh
    init_av = 0
    if fitT:
        teff_range = ast.literal_eval(fits['Trange'])
        init_teff = (teff_range[0] + teff_range[1]) / 2
    else:
        init_teff = 5000
        teff_range = None
    if fitLG:
        logg_range = ast.literal_eval(fits['Loggrange'])
    else:
        logg_range = None
    if fitFEH:
        feh_range = ast.literal_eval(fits['Fehrange'])
    else:
        feh_range = None
    if fitAV:
        av_range = ast.literal_eval(fits['Avrange'])
    else:
        av_range = None

    init_vals = [init_teff, init_logg, init_feh, init_av]
    ranges = [teff_range, logg_range, feh_range, av_range]
    fitflags = [fitT, fitLG, fitFEH, fitAV]
    return init_vals, ranges, fitflags, model

In [ ]:
def save_photometry(starid, phot_obj, out_dir, verbose=False):
    try:
        os.makedirs(out_dir, exist_ok=True)
        print(f"Directory '{out_dir}' created successfully or already exists.")
    except OSError as e:
        print(f"Error creating directory {out_dir}: {e}")

    f = Fitter()
    f.star = phot_obj
    f.out_folder = out_dir
    f.star.save_mags(f.out_folder + '/' + starid.replace(" ", ""))
    if verbose:
        print('File saved:', os.getcwd() + '/' + f.out_folder + '/' + starid.replace(" ", "") + 'mags.dat')
    fn = os.getcwd() + '/' + f.out_folder + '/' + starid + 'mags.dat'
    return fn

In [ ]:
def create_photometry_file(star_id, out_dir, verbose = False):
    starname = star_id
    star = StellarParams()
    ra_deg, dec_deg, ra_hms, dec_hms = pull_coords(starname, star, verbose = verbose)
    gaia_id = pull_gaia_id(starname, star, verbose = verbose)
    photometry = extract_photometry(starname, star, verbose = verbose)
    out_dir = out_dir
    if verbose:
        print(f"Extracted photometry for {starname}")
        
    filename = save_photometry(starname, photometry, out_dir, verbose = verbose)

    return filename

In [ ]:
def sed_process_star(star_name, data_dir, output_dir, stellar_param_dict, fitting_param_dict, unit, set_axis, image_ext, result_rows, diam_rows, 
                     uselatex, logplot, fbol_lam, own_photometry = False, verbose = False):
    
    star_id = extract_id(star_name)
    print("--------------------------------------------------")
    print(f"Starting processing for {star_name}")
    if star_id is None:
        star_id = star_name

    if own_photometry:
        if verbose:
            print("Finding user generated photometry files")
        files = find_files_for_star(star_id, data_dir)
        phot_data = read_in_photometry(files[0])
        if verbose:
            print(f"Files found for {star_name}:", files)
        if not files:
            print(f"No files found for {star_name} ({star_id})")
            return
    if not own_photometry:
        if verbose:
            print(f"Extracting photometry for {star_name}")
        files = create_photometry_file(star_name, data_dir, verbose = verbose)
        phot_data = read_in_photometry(files)
        
    star = StellarParams()
    params = stellar_param_dict.get(star_name, {})
    for param, value in params.items():
        setattr(star, param, value)
    ra_deg, dec_deg, ra_hms, dec_hms = pull_coords(star_name, star, verbose = verbose)
    gaia_id = pull_gaia_id(star_name, star, verbose = verbose)
    D, dD = distances(star_name, verbose = verbose)
    star.dist = D
    star.dist_err = dD

    init_values, fit_ranges, fit_flags, model = convert_fit_params(star_name, star, fitting_param_dict, verbose = False)

    sed_fit = fit_sed(phot_data, star, init_values, model, teffrange = fit_ranges[0], loggrange = fit_ranges[1], 
                      fehrange = fit_ranges[2], avrange = fit_ranges[3], fitT = fit_flags[0], 
                      fit_logg = fit_flags[1], fit_feh = fit_flags[2], fit_av = fit_flags[3], verbose = verbose)
    
    
    fbol, fbol_err = calc_fbol(star, sed_fit, unit = unit, verbose = verbose)

    sed_results, diam_resuls = write_results(star_name, sed_fit, star, output_dir, result_rows, diam_rows)
    
    plot_dir = os.path.join(output_dir, "plots")
    os.makedirs(plot_dir, exist_ok = True)

    
    star_title = convert_names_to_latex([star_name])
    
    fig, _ = plot_sed(sed_fit, 
                      unit = unit, 
                      logplot = logplot, 
                      fbol_lam = fbol_lam, 
                      set_axis = set_axis,
                      title = star_title[0],
                      uselatex = uselatex, 
                      verbose = verbose)
    
    save_plot(fig, plot_dir, star_name.replace(" ", ""), "SEDfit", image_ext)

    print(f"Finished processing {star_name}")

In [ ]:
filename = '/mnt/c/Users/oxfor/Research/rsadpy/tests/test_data/photometry/allphot_test.csv'
df = pd.read_csv('/mnt/c/Users/oxfor/Research/rsadpy/tests/test_data/photometry/allphot_test.csv')

In [ ]:
df

In [ ]:
starname = 'HD 1461'
star = StellarParams()
ra_deg, dec_deg, ra_hms, dec_hms = pull_coords(starname, star, verbose = True)
gaia_id = pull_gaia_id(starname, star, verbose = True)

In [ ]:
def write_results(starname, sed_obj, star,  out_dir, rows_for_results, rows_for_diams):
    star_teff = round(sed_obj.getteff()[0], 2)
    star_av = round(sed_obj.getav(), 4)
    star_rad = round(sed_obj.getr()[0], 3)
    star_fbol = star.fbol
    star_fbol_err = star.fbol_err
    star_logg = star.logg
    star_logg_err = star.logg_err
    star_feh = star.feh
    star_feh_err = star.feh_err

    rows_for_diams.append({
        "Star": starname, 
        "fbol": star_fbol,
        "dfbol": star_fbol_err,
        "logg": star_logg,
        "dlogg": star_logg_err,
        "feh": star_feh,
        "dfeh": star_feh_err
    })

    rows_for_results.append({
        "Star": starname, 
        "SED Teff": star_teff,
        "SED Av": star_av,
        "SED Radius": star_rad
    })

    return rows_for_diams, rows_for_results
    
    

In [ ]:
star_name = 'HD 1461'
data_dir = "/mnt/c/Users/oxfor/Research/rsadpy/tests/tests/test_data/photometry"
output_dir = data_dir
ids, params, fits = get_stellar_params(filename) 
model = 'btsettl'
unit = 'AA'
set_axis = None
image_ext = '.jpg'
logplot = True
fbol_lam = True
uselatex = False
process_star(star_name, data_dir, output_dir, params, fits, model, unit, set_axis, image_ext, uselatex,
             logplot, fbol_lam, own_photometry = True, verbose = True)

In [ ]:
def write_table(df, out_dir, out_file):
    #####################################################
    # Function: write_latex_table                       #
    # Inputs: df -> dataframe to write to table         #
    #         out_file -> output tex file               #
    # Outputs: writes to a latex file                   #
    # What it does:                                     #
    #       1. Opens the out_file                       #
    #       2. Writes to the file                       #
    #####################################################
    os.chdir(out_dir)
    df.to_csv(out_file, sep='\t', index=False)

In [ ]:
def sed_batchmode(starfile, data_dir, out_dir, res_out, diam_out, unit, set_axis, image_ext, uselatex, logplot, fbol_lam, own_photometry, verbose = False):
    os.chdir(data_dir)
    star_names, star_params, fit_params = get_stellar_params(starfile)
    res_rows = []
    diam_rows = []
    count = 0
    for star_name in star_names:
        sed_process_star(star_name, data_dir, out_dir, star_params, fit_params, unit, set_axis, image_ext, res_rows, diam_rows, uselatex, 
                         logplot, fbol_lam, own_photometry = own_photometry, verbose = verbose)
        count += 1

    res_df = pd.DataFrame(res_rows)
    diam_df = pd.DataFrame(diam_rows)

    write_table(res_df, out_dir, res_out)
    write_table(diam_df, out_dir, diam_out)
    
    print(f"Batch complete. Fit {count} stars. Plots in {os.path.join(out_dir, 'plots')}, SED fit results in {res_out}, File for diameter fitting in {diam_out}")


In [ ]:
data_dir = "/mnt/c/Users/oxfor/Research/rsadpy/tests/tests/test_data/photometry"
param_file = '/mnt/c/Users/oxfor/Research/rsadpy/tests/test_data/photometry/allphot_test.csv'
out_dir = "/mnt/c/Users/oxfor/Research/rsadpy/tests/tests/test_data/photometry"
res_out = "SEDresults.txt"
diam_out = "stardatafordiams.txt"
unit = 'AA'
set_axis = None
image_ext = '.jpg'
uselatex = False
logplot = True
fbol_lam = True
own_photometry = True
sed_batchmode(param_file, data_dir, out_dir, res_out, diam_out, unit, set_axis, image_ext, uselatex,
              logplot, fbol_lam, own_photometry, verbose = True)

In [ ]:
df1, df2 = sed_batchmode(param_file, data_dir, out_dir, res_out, diam_out, unit, set_axis, image_ext, uselatex,
              logplot, fbol_lam, own_photometry, verbose = True)

In [ ]:
df1

In [ ]:
df2